In [1]:
# ==========================================================
# CELDA 1 — Instalar dependencias necesarias en Google Colab
# ==========================================================

!pip install -q playwright nest_asyncio pandas openpyxl
!playwright install chromium
!playwright install-deps chromium

print("Dependencias instaladas correctamente.")

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [2]:
# ==========================================================
# CELDA 2 — Importaciones y configuración global
# ==========================================================

import re
import json
import asyncio
import logging
import nest_asyncio
import pandas as pd
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional, List, Dict
from google.colab import drive
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

nest_asyncio.apply()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
log = logging.getLogger("scraper_csj")

@dataclass
class Config:
    # Rutas en Google Drive
    carpeta_raiz: str = "/content/drive/MyDrive/TG_Maestria/01_Corpus_Raw/Sentencias/Corte_Suprema"
    carpeta_debug: str = "/content/drive/MyDrive/TG_Maestria/01_Corpus_Raw/Debug_Scraping_CSJ"

    # Timeouts
    timeout_navegacion: int = 90000
    timeout_selector:   int = 30000

    # Pausas (segundos)
    pausa_corta:  float = 1.0
    pausa_media:  float = 2.5
    pausa_larga:  float = 6.0

    # Navegador
    headless:        bool = True
    viewport_width:  int  = 1440
    viewport_height: int  = 900
    user_agent: str = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

CONFIG = Config()
log.info("Configuración cargada")
print(json.dumps(asdict(CONFIG), indent=2, ensure_ascii=False))

{
  "carpeta_raiz": "/content/drive/MyDrive/TG_Maestria/01_Corpus_Raw/Sentencias/Corte_Suprema",
  "carpeta_debug": "/content/drive/MyDrive/TG_Maestria/01_Corpus_Raw/Debug_Scraping_CSJ",
  "timeout_navegacion": 90000,
  "timeout_selector": 30000,
  "pausa_corta": 1.0,
  "pausa_media": 2.5,
  "pausa_larga": 6.0,
  "headless": true,
  "viewport_width": 1440,
  "viewport_height": 900,
  "user_agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}


In [3]:
# ==========================================================
# CELDA 3 — Montar Google Drive y preparar carpetas
# ==========================================================

drive.mount('/content/drive')

RUTA_CSJ      = Path(CONFIG.carpeta_raiz)
RUTA_DEBUG    = Path(CONFIG.carpeta_debug)
RUTA_ARCHIVOS = RUTA_CSJ / "archivos"
RUTA_TABLAS   = RUTA_CSJ / "tablas"

for ruta in [RUTA_CSJ, RUTA_DEBUG, RUTA_ARCHIVOS, RUTA_TABLAS]:
    ruta.mkdir(parents=True, exist_ok=True)
    log.info(f"Carpeta lista: {ruta}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ==========================================================
# CELDA 4 — Diccionario de temas de búsqueda
# ==========================================================

TEMAS = [
    "INTERPRETACION DE LOS CONTRATOS",
    "AUTONOMIA VOLUNTAD PRIVADA",
    "VICIOS DEL CONSENTIMIENTO",
    "CONDICION RESOLUTORIA TACITA",
    "EXCEPCION DE CONTRATO NO CUMPLIDO",
    "BUENA FE OBJETIVA CONTRACTUAL",
    "CLAUSULA PENAL",
    "LESION ENORME",
    "SIMULACION ABSOLUTA",
    "TEORIA DE LA IMPREVISION",
]

# Máximo de resultados a descargar por tema (ajustar según necesidad)
MAX_POR_TEMA = 50

print(f"Temas configurados: {len(TEMAS)}")
for i, t in enumerate(TEMAS, 1):
    print(f"  {i:02d}. {t}")

Temas configurados: 10
  01. INTERPRETACION DE LOS CONTRATOS
  02. AUTONOMIA VOLUNTAD PRIVADA
  03. VICIOS DEL CONSENTIMIENTO
  04. CONDICION RESOLUTORIA TACITA
  05. EXCEPCION DE CONTRATO NO CUMPLIDO
  06. BUENA FE OBJETIVA CONTRACTUAL
  07. CLAUSULA PENAL
  08. LESION ENORME
  09. SIMULACION ABSOLUTA
  10. TEORIA DE LA IMPREVISION


In [4]:
# ==========================================================
# CELDA 5 — URL base y selectores confirmados por DOM
# ==========================================================

BASE_URL = "https://consultajurisprudencial.ramajudicial.gov.co/WebRelatoria/csj/index.xhtml"

SEL = {
    # --- Formulario búsqueda ---
    "tema_input": "#searchForm\\:temaInput",
    "buscar_button": "#searchForm\\:searchButton",
    "nueva_busqueda": "#searchForm\\:j_idt247",

    # --- Sala (selectCheckboxMenu PrimeFaces) ---
    "sala_trigger": "#searchForm\\:scivil .ui-selectcheckboxmenu-trigger",
    "sala_panel_id": "searchForm:scivil_panel",
    "sala_label": "#searchForm\\:scivil .ui-selectcheckboxmenu-label",

    # --- Radios Asuntos ---
    "radio_asuntos_sala": "#searchForm\\:tutelaselect\\:0",
    "label_asuntos_sala": 'label[for="searchForm:tutelaselect:0"]',

    # --- Radios Relevancia ---
    "radio_relevantes": "#searchForm\\:relevanteselect\\:0",
    "label_relevantes": 'label[for="searchForm:relevanteselect:0"]',

    # --- Formulario resultados ---
"result_form": "#resultForm",
"result_panel_text": "text=Resultado:",
"resultado_primero_texto": "text=RELEVANTE",

# --- Checkboxes de resultados (sin ids dinámicos) ---
"checkboxes_resultados": "#resultForm .ui-chkbox-box",
"checkboxes_inputs": "#resultForm input[type='checkbox']",

# --- Descarga splitButton ---
"btn_descargar_main": "#resultForm\\:j_idt275_button",
"btn_descargar_menu": "#resultForm\\:j_idt275_menuButton",
"menu_descargar": "#resultForm\\:j_idt275_menu",
"opcion_pdf": "#resultForm\\:j_idt278",

    # --- Paginación ---
    "btn_primero": "#resultForm\\:j_idt256",
    "btn_anterior": "#resultForm\\:j_idt257",
    "btn_siguiente": "#resultForm\\:j_idt258",
    "btn_ultimo": "#resultForm\\:j_idt259",

    # --- Seleccionar todos ---
    "btn_select_all": "#resultForm\\:selectAllButton",
}

print("✓ Selectores unificados cargados")
for k, v in SEL.items():
    print(f" {k}: {v}")

✓ Selectores unificados cargados
 tema_input: #searchForm\:temaInput
 buscar_button: #searchForm\:searchButton
 nueva_busqueda: #searchForm\:j_idt247
 sala_trigger: #searchForm\:scivil .ui-selectcheckboxmenu-trigger
 sala_panel_id: searchForm:scivil_panel
 sala_label: #searchForm\:scivil .ui-selectcheckboxmenu-label
 radio_asuntos_sala: #searchForm\:tutelaselect\:0
 label_asuntos_sala: label[for="searchForm:tutelaselect:0"]
 radio_relevantes: #searchForm\:relevanteselect\:0
 label_relevantes: label[for="searchForm:relevanteselect:0"]
 result_form: #resultForm
 result_panel_text: text=Resultado:
 resultado_primero_texto: text=RELEVANTE
 checkboxes_resultados: #resultForm .ui-chkbox-box
 checkboxes_inputs: #resultForm input[type='checkbox']
 btn_descargar_main: #resultForm\:j_idt275_button
 btn_descargar_menu: #resultForm\:j_idt275_menuButton
 menu_descargar: #resultForm\:j_idt275_menu
 opcion_pdf: #resultForm\:j_idt278
 btn_primero: #resultForm\:j_idt256
 btn_anterior: #resultForm\:j_id

In [5]:
# ==========================================================
# CELDA 6 — Utilidades de texto y archivos
# ==========================================================

def clean_text(text: Optional[str]) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def slugify(text: str) -> str:
    text = clean_text(text).lower()
    text = re.sub(r"[^a-z0-9áéíóúñ]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")[:100]


def limpiar_nombre_archivo(texto: str) -> str:
    return re.sub(r'[\\/*?:"<>|]+', "", texto).strip()[:120]


def parse_resultado(raw: str) -> Dict[str, str]:
    """Extrae campos estructurados del texto visible de un resultado."""
    campos = {
        "sala":               r"(SALA DE [^\n]+)",
        "etiqueta":           r"\b(RELEVANTE|GACETA JUDICIAL)\b",
        "id":                 r"ID:\s*([^\n]+)",
        "numero_proceso":     r"NÚMERO DE PROCESO:\s*([^\n]+)",
        "numero_providencia": r"NÚMERO DE PROVIDENCIA:\s*([^\n]+)",
        "clase_actuacion":    r"CLASE DE ACTUACIÓN:\s*([^\n]+)",
        "tipo_providencia":   r"TIPO DE PROVIDENCIA:\s*([^\n]+)",
        "fecha":              r"FECHA:\s*([^\n]+)",
        "ponente":            r"PONENTE:\s*([^\n]+)",
        "tema_resultado":     r"TEMA:\s*([^\n]+)",
        "extracto":           r"EXTRACTO\s*[-–]\s*(.+?)(?=UNIFICACIÓN|$)",
    }
    data: Dict[str, str] = {}
    for key, pattern in campos.items():
        m = re.search(pattern, raw, re.MULTILINE | re.DOTALL)
        data[key] = clean_text(m.group(1)) if m else ""
    data["texto_completo"] = clean_text(raw)
    return data


print("Utilidades cargadas")

Utilidades cargadas


In [6]:
# ==========================================================
# CELDA 7 — Helpers de Playwright
# ==========================================================

async def guardar_debug(page, prefijo: str):
    await page.screenshot(
        path=str(RUTA_DEBUG / f"{prefijo}.png"),
        full_page=True
    )
    (RUTA_DEBUG / f"{prefijo}.html").write_text(
        await page.content(), encoding="utf-8"
    )
    log.info(f"Debug guardado: {prefijo}")


async def safe_text(locator) -> str:
    try:
        return clean_text(await locator.inner_text())
    except Exception:
        return ""


async def wait_fill(page, selector: str, value: str):
    await page.wait_for_selector(selector, timeout=CONFIG.timeout_selector)
    await page.locator(selector).fill(value)


async def wait_click(page, selector: str):
    await page.wait_for_selector(selector, timeout=CONFIG.timeout_selector)
    await page.locator(selector).click()


async def iniciar_contexto(headless: bool = None):
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(
        headless=headless if headless is not None else CONFIG.headless
    )
    ctx = await browser.new_context(
        accept_downloads=True,
        user_agent=CONFIG.user_agent,
        viewport={"width": CONFIG.viewport_width, "height": CONFIG.viewport_height},
    )
    page = await ctx.new_page()
    page.set_default_timeout(CONFIG.timeout_selector)
    return pw, browser, ctx, page

print("✓ Helpers listos")

✓ Helpers listos


In [7]:
# ==========================================================
# CELDA 8 — Abrir página y aplicar filtros
# ==========================================================

async def abrir_pagina(page):
    await page.goto(
        BASE_URL,
        wait_until="domcontentloaded",
        timeout=CONFIG.timeout_navegacion
    )
    await page.wait_for_selector(
        SEL["tema_input"],
        timeout=CONFIG.timeout_selector
    )
    log.info(f"Página cargada: {page.url}")


async def seleccionar_sala_civil(page) -> bool:
    """
    Selecciona 'SALA DE CASACIÓN CIVIL' en el selectCheckboxMenu.
    Estructura confirmada por diagnóstico:
      - Panel id: searchForm:scivil_panel
      - Items: >
      - Checkbox: <div class="ui-chkbox-box"> (el input interno es readonly)
      - Label: abel>SALA DE CASACIÓN CIVIL</label>
    """
    try:
        # 1) Abrir el panel con clic en el trigger
        await page.locator(SEL["sala_trigger"]).first.click()
        await page.wait_for_timeout(1500)
        log.info("Trigger de sala clickeado")

        # 2) Esperar display:block en el panel
        await page.wait_for_function(
            f"""
            () => {{
                const panel = document.getElementById('{SEL["sala_panel_id"]}');
                return panel && panel.style.display === 'block';
            }}
            """,
            timeout=5000
        )
        log.info("Panel de sala visible")

        # 3) Clic en ui-chkbox-box del item Civil via JS
        resultado = await page.evaluate(
            f"""
            () => {{
                const panel = document.getElementById('{SEL["sala_panel_id"]}');
                if (!panel) return {{ ok: false, msg: 'Panel no encontrado' }};

                const items = panel.querySelectorAll('li.ui-selectcheckboxmenu-item');
                if (items.length === 0) {{
                    return {{
                        ok: false,
                        msg: 'No se encontraron items li',
                        innerHTML: panel.innerHTML.substring(0, 300)
                    }};
                }}

                let civil_clicado = false;
                const detalle = [];

                items.forEach((item, i) => {{
                    const label    = item.querySelector('label');
                    const chk_box  = item.querySelector('.ui-chkbox-box');
                    const texto    = label ? label.textContent.trim() : '';
                    const es_civil = texto.toUpperCase().includes('CIVIL');
                    const checked  = item.classList.contains(
                        'ui-selectcheckboxmenu-checked'
                    );

                    detalle.push({{ i, texto, es_civil, checked }});

                    if (es_civil && !checked && chk_box) {{
                        chk_box.click();
                        civil_clicado = true;
                    }} else if (!es_civil && checked && chk_box) {{
                        chk_box.click();  // desmarcar otras salas
                    }}
                }});

                return {{
                    ok: civil_clicado,
                    msg: civil_clicado
                        ? 'Civil marcado'
                        : 'Civil ya estaba marcado o no encontrado',
                    items: detalle
                }};
            }}
            """
        )

        log.info(f"Selección sala: {resultado}")
        await page.wait_for_timeout(800)

        # 4) Cerrar el panel
        try:
            close = page.locator(
                f"#{SEL['sala_panel_id'].replace(':', '\\:')} "
                ".ui-selectcheckboxmenu-close"
            ).first
            if await close.count() > 0:
                await close.click()
                log.info("Panel cerrado con botón X")
            else:
                await page.locator(SEL["tema_input"]).click()
                log.info("Panel cerrado con clic en tema")
        except Exception:
            await page.keyboard.press("Escape")

        await page.wait_for_timeout(800)

        # 5) Verificar label
        label_actual = await page.evaluate(
            f"""
            () => {{
                const l = document.querySelector(
                    '#{SEL["sala_panel_id"].replace(":", "\\\\:")} '
                    + '.ui-selectcheckboxmenu-label, '
                    + '#searchForm\\\\:scivil .ui-selectcheckboxmenu-label'
                );
                return l ? l.textContent.trim() : 'no encontrado';
            }}
            """
        )
        log.info(f"Label sala post-selección: '{label_actual}'")
        return resultado.get("ok", False) or "civil" in label_actual.lower()

    except Exception as e:
        log.error(f"Error seleccionar_sala_civil: {e}")
        return False


async def aplicar_filtros(page, tema: str):
    # 1) Tema
    await wait_fill(page, SEL["tema_input"], tema)
    await page.wait_for_timeout(600)
    log.info(f"Tema: '{tema}'")

    # 2) Sala Civil
    sala_ok = await seleccionar_sala_civil(page)
    if not sala_ok:
        log.warning("⚠ Sala Civil no confirmada")

    # 3) Asuntos de Sala
    try:
        if not await page.locator(SEL["radio_asuntos_sala"]).is_checked():
            await page.locator(SEL["label_asuntos_sala"]).click()
            await page.wait_for_timeout(500)
            log.info("'Asuntos de Sala' marcado")
        else:
            log.info("'Asuntos de Sala' ya marcado ✓")
    except Exception as e:
        log.warning(f"Asuntos de Sala: {e}")

    # 4) Relevantes
    try:
        if not await page.locator(SEL["radio_relevantes"]).is_checked():
            await page.locator(SEL["label_relevantes"]).click()
            await page.wait_for_timeout(500)
            log.info("'Relevantes' marcado")
        else:
            log.info("'Relevantes' ya marcado ✓")
    except Exception as e:
        log.warning(f"Relevantes: {e}")

    await page.wait_for_timeout(500)

    # 5) Log estado final
    try:
        sala_label = await page.evaluate("""
            () => {
                const l = document.querySelector(
                    '#searchForm\\\\:scivil .ui-selectcheckboxmenu-label'
                );
                return l ? l.textContent.trim() : 'no encontrado';
            }
        """)
        asuntos_ok = await page.locator(SEL["radio_asuntos_sala"]).is_checked()
        relev_ok   = await page.locator(SEL["radio_relevantes"]).is_checked()
        log.info(
            f"Estado final → sala='{sala_label}' | "
            f"asuntos_sala={asuntos_ok} | relevantes={relev_ok}"
        )
    except Exception:
        pass

print("✓ Filtros definidos")

✓ Filtros definidos


In [ ]:
# ==========================================================
# CELDA 8B — Ejecutar búsqueda y descargar PDF
# ==========================================================

async def ejecutar_busqueda(page):
    await wait_click(page, SEL["buscar_button"])
    await page.wait_for_timeout(5000)

    await page.wait_for_selector(
        SEL["result_form"],
        timeout=CONFIG.timeout_selector
    )

    texto = await safe_text(page.locator(SEL["result_form"]))
    log.info("Búsqueda ejecutada — panel de resultados detectado")
    return texto


async def seleccionar_primer_resultado(page):
    """
    Selecciona el primer checkbox visible dentro del panel de resultados,
    evitando ids dinámicos j_idtXXX.
    """
    await page.wait_for_selector(SEL["result_form"], timeout=CONFIG.timeout_selector)
    await page.wait_for_timeout(2000)

    candidatos = page.locator(SEL["checkboxes_resultados"])
    total = await candidatos.count()
    log.info(f"Checkboxes detectados en resultForm: {total}")

    if total == 0:
        raise Exception("No se encontraron checkboxes dentro de resultForm")

    for i in range(total):
        box = candidatos.nth(i)
        try:
            if await box.is_visible():
                clases = await box.get_attribute("class") or ""
                if "ui-state-disabled" in clases:
                    continue

                if "ui-state-active" not in clases:
                    await box.click(force=True)
                    await page.wait_for_timeout(1500)
                    log.info(f"Primer checkbox útil seleccionado en índice {i}")
                else:
                    log.info(f"Checkbox índice {i} ya estaba seleccionado")
                return i
        except Exception:
            continue

    raise Exception("Se encontraron checkboxes, pero ninguno fue clickeable")


async def descargar_pdf_resultado_actual(page, nombre_base: str = "sentencia"):
    await page.wait_for_selector(
        SEL["btn_descargar_menu"],
        timeout=CONFIG.timeout_selector
    )

    await page.locator(SEL["btn_descargar_menu"]).click()
    await page.wait_for_timeout(1500)

    await page.wait_for_selector(
        SEL["menu_descargar"],
        timeout=CONFIG.timeout_selector
    )
    await page.wait_for_selector(
        SEL["opcion_pdf"],
        timeout=CONFIG.timeout_selector
    )

    async with page.expect_download(timeout=CONFIG.timeout_navegacion) as download_info:
        await page.locator(SEL["opcion_pdf"]).click()

    download = await download_info.value
    suggested = download.suggested_filename or f"{limpiar_nombre_archivo(nombre_base)}.pdf"

    if not suggested.lower().endswith(".pdf"):
        suggested = f"{suggested}.pdf"

    destino = RUTA_ARCHIVOS / limpiar_nombre_archivo(suggested)
    await download.save_as(str(destino))

    log.info(f"PDF descargado: {destino}")
    return destino


print("Funciones actualizadas para selección robusta y descarga PDF")

✓ Funciones actualizadas para selección robusta y descarga PDF


In [10]:
# ==========================================================
# CELDA 9 — Prueba completa con descarga PDF
# ==========================================================

async def prueba_busqueda(tema: str, headless: bool = True):
    pw, browser, ctx, page = await iniciar_contexto(headless=headless)
    try:
        await abrir_pagina(page)
        await aplicar_filtros(page, tema)

        texto = await ejecutar_busqueda(page)
        await guardar_debug(page, f"prueba_{slugify(tema)[:40]}_resultados")

        contador_match = re.search(r"Resultado:\s*\d+\s*/\s*\d+", texto, re.IGNORECASE)
        contador = contador_match.group(0) if contador_match else "resultado no detectado"

        sala_match = re.search(r"SALA DE CASACIÓN\s+[A-ZÁÉÍÓÚÑ]+", texto, re.IGNORECASE)
        sala = sala_match.group(0) if sala_match else "sala no detectada"

        print("\n--- CONTADOR ---")
        print(contador)

        print("\n--- SALA DETECTADA ---")
        print(sala)

        print("\n--- PRIMEROS 1200 CARACTERES ---")
        print(texto[:1200])

        idx = await seleccionar_primer_resultado(page)
        print(f"\nCheckbox usado: índice {idx}")

        await guardar_debug(page, f"prueba_{slugify(tema)[:40]}_seleccionado")

        pdf_path = await descargar_pdf_resultado_actual(
            page,
            nombre_base=f"{slugify(tema)}_resultado_1"
        )

        print("\n--- PDF DESCARGADO ---")
        print(pdf_path)

        await guardar_debug(page, f"prueba_{slugify(tema)[:40]}_pdf_descargado")

    finally:
        await browser.close()
        await pw.stop()


await prueba_busqueda("AUTONOMIA VOLUNTAD PRIVADA", headless=True)

TimeoutError: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("#searchForm\\:temaInput") to be visible


In [ ]:
# ==========================================================
# CELDA 10 — Temas semilla derivados del Golden Set
# ==========================================================

TEMAS_GOLDEN = [
    "INTERPRETACION DE LOS CONTRATOS",
    "AUTONOMIA DE LA VOLUNTAD PRIVADA",
    "VICIOS DEL CONSENTIMIENTO",
    "ERROR EN EL CONSENTIMIENTO",
    "CLAUSULA PENAL",
    "CONDICION RESOLUTORIA TACITA",
    "EXCEPCION DE CONTRATO NO CUMPLIDO",
    "BUENA FE OBJETIVA CONTRACTUAL",
    "TEORIA DE LA IMPREVISION",
    "LESION ENORME",
    "SIMULACION ABSOLUTA",
    "SIMULACION RELATIVA",
    "INCUMPLIMIENTO CONTRACTUAL",
    "RESOLUCION DEL CONTRATO",
    "INTERPRETACION DE CLAUSULAS CONTRACTUALES",
    "OBLIGACIONES CONDICIONALES",
    "OBLIGACIONES A PLAZO",
    "RESPONSABILIDAD CONTRACTUAL",
    "ARRAS",
    "FUERZA MAYOR",
    "CASO FORTUITO",
    "MORA DEL DEUDOR",
    "MORA DEL ACREEDOR",
    "NULIDAD RELATIVA",
    "OBJETO DEL CONTRATO",
    "CAUSA DEL CONTRATO",
    "PROMESA DE COMPRAVENTA",
    "MANDATO",
    "COMPRAVENTA",
    "ARRENDAMIENTO",
    "MUTUO",
    "CONTRATO DE OBRA",
    "VICIOS OCULTOS",
    "PACTO DE RESERVA DE DOMINIO",
    "DACION EN PAGO",
    "NOVACION",
    "CESION DE CREDITO",
    "CESION DE POSICION CONTRACTUAL",
    "SUBROGACION",
    "OBLIGACIONES SOLIDARIAS",
]

# Quita duplicados conservando orden
TEMAS_BUSQUEDA = list(dict.fromkeys(TEMAS_GOLDEN))

META_DESCARGAS = 2000
print(f"Temas de búsqueda preparados: {len(TEMAS_BUSQUEDA)}")
print(f"Meta total de PDFs: {META_DESCARGAS}")
for i, t in enumerate(TEMAS_BUSQUEDA[:15], 1):
    print(f"{i:02d}. {t}")

In [ ]:
# ==========================================================
# CELDA 11 — Estado del scraping y control de duplicados
# ==========================================================

ESTADO_PATH = RUTA_TABLAS / "estado_scraping_csj.json"
CSV_RESULTADOS_PATH = RUTA_TABLAS / "metadata_sentencias_csj.csv"
CSV_FALLOS_PATH = RUTA_TABLAS / "fallos_scraping_csj.csv"

def cargar_csv_seguro(path: Path) -> pd.DataFrame:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            return pd.DataFrame()
    return pd.DataFrame()

def cargar_estado():
    if ESTADO_PATH.exists():
        try:
            return json.loads(ESTADO_PATH.read_text(encoding="utf-8"))
        except Exception:
            pass
    return {
        "temas_completados": [],
        "total_descargas": 0,
        "total_fallos": 0,
    }

def guardar_estado(estado: dict):
    ESTADO_PATH.write_text(
        json.dumps(estado, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

df_existente = cargar_csv_seguro(CSV_RESULTADOS_PATH)
df_fallos = cargar_csv_seguro(CSV_FALLOS_PATH)
estado = cargar_estado()

IDS_DESCARGADOS = set()
PDFS_DESCARGADOS = set()

if not df_existente.empty:
    if "id" in df_existente.columns:
        IDS_DESCARGADOS = set(df_existente["id"].dropna().astype(str).str.strip())
    if "ruta_pdf" in df_existente.columns:
        PDFS_DESCARGADOS = set(df_existente["ruta_pdf"].dropna().astype(str).str.strip())

print("Estado cargado")
print(f"- temas completados: {len(estado['temas_completados'])}")
print(f"- total_descargas: {estado['total_descargas']}")
print(f"- ids descargados: {len(IDS_DESCARGADOS)}")

In [ ]:
# ==========================================================
# CELDA 12 — Parseo, normalización y persistencia
# ==========================================================

def construir_nombre_pdf(data: dict, tema_busqueda: str) -> str:
    partes = [
        data.get("numero_providencia", ""),
        data.get("id", ""),
        data.get("fecha", ""),
        tema_busqueda,
    ]
    base = "_".join([slugify(p) for p in partes if clean_text(p)])
    if not base:
        base = slugify(tema_busqueda) or "sentencia"
    return limpiar_nombre_archivo(base[:120]) + ".pdf"

def consolidar_registro(data: dict, tema_busqueda: str, ruta_pdf: str = "") -> dict:
    return {
        "tema_busqueda": tema_busqueda,
        "sala": data.get("sala", ""),
        "etiqueta": data.get("etiqueta", ""),
        "id": data.get("id", ""),
        "numero_proceso": data.get("numero_proceso", ""),
        "numero_providencia": data.get("numero_providencia", ""),
        "clase_actuacion": data.get("clase_actuacion", ""),
        "tipo_providencia": data.get("tipo_providencia", ""),
        "fecha": data.get("fecha", ""),
        "ponente": data.get("ponente", ""),
        "tema_resultado": data.get("tema_resultado", ""),
        "extracto": data.get("extracto", ""),
        "texto_completo": data.get("texto_completo", ""),
        "ruta_pdf": ruta_pdf,
    }

def append_csv(path: Path, row: dict):
    df_row = pd.DataFrame([row])
    if path.exists():
        df_old = pd.read_csv(path)
        df_new = pd.concat([df_old, df_row], ignore_index=True)
    else:
        df_new = df_row
    df_new.to_csv(path, index=False)

def registrar_fallo(tema_busqueda: str, etapa: str, error: str, extra: dict = None):
    row = {
        "tema_busqueda": tema_busqueda,
        "etapa": etapa,
        "error": clean_text(str(error)),
    }
    if extra:
        row.update(extra)
    append_csv(CSV_FALLOS_PATH, row)

In [ ]:
# ==========================================================
# CELDA 13 — Lectura del resultado actual y duplicados
# ==========================================================

async def obtener_texto_resultado_actual(page) -> str:
    await page.wait_for_selector(SEL["result_form"], timeout=CONFIG.timeout_selector)
    texto = await safe_text(page.locator(SEL["result_form"]))
    return texto

async def extraer_resultado_actual(page, tema_busqueda: str) -> dict:
    texto = await obtener_texto_resultado_actual(page)
    data = parse_resultado(texto)
    data["tema_busqueda"] = tema_busqueda
    return data

def ya_descargado(data: dict) -> bool:
    _id = clean_text(data.get("id", ""))
    if _id and _id in IDS_DESCARGADOS:
        return True
    return False

def marcar_descargado(data: dict, ruta_pdf: str):
    _id = clean_text(data.get("id", ""))
    if _id:
        IDS_DESCARGADOS.add(_id)
    if ruta_pdf:
        PDFS_DESCARGADOS.add(ruta_pdf)

In [ ]:
# ==========================================================
# CELDA 14 — Navegación entre resultados
# ==========================================================

async def obtener_contador_resultados(page) -> tuple[int, int]:
    texto = await obtener_texto_resultado_actual(page)
    m = re.search(r"Resultado:\s*(\d+)\s*/\s*(\d+)", texto, re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))
    return 0, 0

async def ir_al_siguiente_resultado(page) -> bool:
    try:
        await page.wait_for_selector(SEL["btn_siguiente"], timeout=CONFIG.timeout_selector)
        btn = page.locator(SEL["btn_siguiente"])
        clases = await btn.get_attribute("class") or ""
        aria_disabled = await btn.get_attribute("aria-disabled")

        if "ui-state-disabled" in clases or aria_disabled == "true":
            return False

        antes = await obtener_contador_resultados(page)
        await btn.click()
        await page.wait_for_timeout(2500)

        for _ in range(10):
            despues = await obtener_contador_resultados(page)
            if despues != antes and despues != (0, 0):
                return True
            await page.wait_for_timeout(700)

        return True
    except Exception:
        return False

In [ ]:
# ==========================================================
# CELDA 15 — Descargar el PDF del resultado actual
# ==========================================================

async def descargar_resultado_actual(page, tema_busqueda: str) -> dict | None:
    data = await extraer_resultado_actual(page, tema_busqueda)

    if ya_descargado(data):
        log.info(f"Duplicado omitido por ID: {data.get('id', '')}")
        return None

    await seleccionar_primer_resultado(page)
    await page.wait_for_timeout(1200)

    nombre_pdf = construir_nombre_pdf(data, tema_busqueda)
    destino = RUTA_ARCHIVOS / nombre_pdf

    if destino.exists():
        registro = consolidar_registro(data, tema_busqueda, str(destino))
        marcar_descargado(data, str(destino))
        log.info(f"Archivo ya existente, se reutiliza: {destino.name}")
        return registro

    try:
        temp_path = await descargar_pdf_resultado_actual(
            page,
            nombre_base=nombre_pdf.replace(".pdf", "")
        )

        temp_path = Path(temp_path)
        if temp_path != destino:
            temp_path.rename(destino)

        registro = consolidar_registro(data, tema_busqueda, str(destino))
        marcar_descargado(data, str(destino))

        append_csv(CSV_RESULTADOS_PATH, registro)
        estado["total_descargas"] += 1
        guardar_estado(estado)

        log.info(f"Descarga OK: {destino.name}")
        return registro

    except Exception as e:
        estado["total_fallos"] += 1
        guardar_estado(estado)
        registrar_fallo(
            tema_busqueda=tema_busqueda,
            etapa="descarga_pdf",
            error=str(e),
            extra={
                "id": data.get("id", ""),
                "numero_providencia": data.get("numero_providencia", ""),
                "numero_proceso": data.get("numero_proceso", ""),
            }
        )
        log.error(f"Error descargando PDF: {e}")
        return None

In [ ]:
# ==========================================================
# CELDA 16 — Procesar un tema completo
# ==========================================================

async def procesar_tema(page, tema_busqueda: str, max_por_tema: int = 100):
    log.info(f"=== Tema: {tema_busqueda} ===")

    try:
        await abrir_pagina(page)
        await aplicar_filtros(page, tema_busqueda)
        await ejecutar_busqueda(page)
        await guardar_debug(page, f"tema_{slugify(tema_busqueda)[:40]}_inicio")
    except Exception as e:
        registrar_fallo(tema_busqueda, "busqueda_inicial", str(e))
        return {
            "tema": tema_busqueda,
            "descargadas": 0,
            "procesadas": 0,
            "errores": 1,
        }

    descargadas = 0
    procesadas = 0
    errores = 0
    vistos_ids = set()

    for paso in range(max_por_tema):
        try:
            actual, total = await obtener_contador_resultados(page)
            data = await extraer_resultado_actual(page, tema_busqueda)

            firma = clean_text(data.get("id", "")) or f"{actual}_{clean_text(data.get('numero_providencia', ''))}"
            if firma in vistos_ids:
                log.warning("Resultado repetido detectado, se detiene el tema")
                break
            vistos_ids.add(firma)

            registro = await descargar_resultado_actual(page, tema_busqueda)
            procesadas += 1
            if registro is not None:
                descargadas += 1

            log.info(f"[{tema_busqueda}] Resultado {actual}/{total} | descargadas tema={descargadas}")

            if total and actual >= total:
                break

            ok = await ir_al_siguiente_resultado(page)
            if not ok:
                break

        except Exception as e:
            errores += 1
            registrar_fallo(
                tema_busqueda,
                "iteracion_resultado",
                str(e),
                extra={"paso": paso + 1}
            )
            break

    if tema_busqueda not in estado["temas_completados"]:
        estado["temas_completados"].append(tema_busqueda)
        guardar_estado(estado)

    return {
        "tema": tema_busqueda,
        "descargadas": descargadas,
        "procesadas": procesadas,
        "errores": errores,
    }

In [ ]:
# ==========================================================
# CELDA 17 — Ejecución masiva hasta meta global
# ==========================================================

async def ejecutar_lote(
    temas: list[str],
    meta_total: int = 2000,
    max_por_tema: int = 120,
    headless: bool = True,
    reintentar_omitidos: bool = False,
):
    resumen = []

    pendientes = temas
    if not reintentar_omitidos:
        pendientes = [t for t in temas if t not in estado["temas_completados"]]

    log.info(f"Temas pendientes: {len(pendientes)}")

    pw, browser, ctx, page = await iniciar_contexto(headless=headless)
    try:
        for i, tema in enumerate(pendientes, 1):
            if estado["total_descargas"] >= meta_total:
                log.info("Meta global alcanzada")
                break

            print(f"\n===== TEMA {i}/{len(pendientes)}: {tema} =====")
            try:
                salida = await procesar_tema(page, tema, max_por_tema=max_por_tema)
                resumen.append(salida)
            except Exception as e:
                registrar_fallo(tema, "tema_completo", str(e))
                resumen.append({
                    "tema": tema,
                    "descargadas": 0,
                    "procesadas": 0,
                    "errores": 1,
                })

            guardar_estado(estado)
            pd.DataFrame(resumen).to_csv(RUTA_TABLAS / "resumen_lote_csj.csv", index=False)

            print("Resumen parcial:")
            print(pd.DataFrame(resumen).tail(5))
            print(f"Total descargas acumuladas: {estado['total_descargas']}")

    finally:
        await browser.close()
        await pw.stop()

    df_resumen = pd.DataFrame(resumen)
    df_resumen.to_csv(RUTA_TABLAS / "resumen_lote_csj.csv", index=False)
    return df_resumen

In [ ]:
# ==========================================================
# CELDA 18 — Lanzar scraping masivo (corregida sin await final)
# ==========================================================

async def main():
    df_resumen = await ejecutar_lote(
    temas=TEMAS_BUSQUEDA,   # todos los 40 temas
    meta_total=2000,         # meta global
    max_por_tema=120,        # tope por tema
    headless=True,
    reintentar_omitidos=False,
)

    print("\n=== RESUMEN FINAL DEL LOTE DE PRUEBA ===")
    print(df_resumen)

    print("\n=== ESTADO ACUMULADO ===")
    print(f"Total PDFs descargados según estado: {estado['total_descargas']}")
    print(f"Total fallos según estado: {estado['total_fallos']}")
    print(f"Metadata CSV: {CSV_RESULTADOS_PATH}")
    print(f"Fallos CSV: {CSV_FALLOS_PATH}")
    print(f"Resumen lote: {RUTA_TABLAS / 'resumen_lote_csj.csv'}")

    return df_resumen

df_resumen = asyncio.get_event_loop().run_until_complete(main())

In [ ]:
# ==========================================================
# CELDA 19 — Auditoría de resultados
# ==========================================================

df_meta = cargar_csv_seguro(CSV_RESULTADOS_PATH)
df_fallos = cargar_csv_seguro(CSV_FALLOS_PATH)
df_resumen = cargar_csv_seguro(RUTA_TABLAS / "resumen_lote_csj.csv")

print("=== AUDITORÍA ===")
print(f"Registros metadata: {len(df_meta)}")
print(f"Fallos registrados: {len(df_fallos)}")
print(f"Temas en resumen: {len(df_resumen)}")

if not df_meta.empty:
    print("\nTop temas por volumen:")
    print(df_meta["tema_busqueda"].value_counts().head(20))

    print("\nTop providencias duplicadas por ID:")
    if "id" in df_meta.columns:
        print(df_meta["id"].value_counts().head(10))

    print("\nRango de fechas:")
    if "fecha" in df_meta.columns:
        print(df_meta["fecha"].dropna().head())

In [ ]:
# ==========================================================
# CELDA 20 — Consolidación final para corpus RAG
# ==========================================================

df_meta = cargar_csv_seguro(CSV_RESULTADOS_PATH)

if not df_meta.empty:
    cols_pref = [
        "id", "numero_providencia", "numero_proceso", "fecha",
        "ponente", "tipo_providencia", "clase_actuacion",
        "tema_busqueda", "tema_resultado", "extracto", "ruta_pdf"
    ]
    cols_pref = [c for c in cols_pref if c in df_meta.columns]

    if "id" in df_meta.columns:
        df_meta["id"] = df_meta["id"].astype(str).str.strip()
        df_meta = df_meta.drop_duplicates(subset=["id"], keep="first")
    elif "ruta_pdf" in df_meta.columns:
        df_meta = df_meta.drop_duplicates(subset=["ruta_pdf"], keep="first")
    else:
        df_meta = df_meta.drop_duplicates()

    df_meta = df_meta.sort_values(by=[c for c in ["tema_busqueda", "fecha"] if c in df_meta.columns])
    FINAL_PATH = RUTA_TABLAS / "corpus_sentencias_csj_rag.csv"
    df_meta.to_csv(FINAL_PATH, index=False)

    print(f"Corpus consolidado: {FINAL_PATH}")
    print(f"Total único de sentencias: {len(df_meta)}")
    print(df_meta[cols_pref].head(10))
else:
    print("No hay metadata para consolidar todavía.")

In [8]:
# ==========================================================
# CELDA 21 — Parser corregido para deduplicación robusta
# ==========================================================

def parse_resultado(raw: str) -> Dict[str, str]:
    raw = raw or ""
    raw_clean = clean_text(raw)

    patrones = {
        "sala": r"(SALA DE CASACIÓN [A-ZÁÉÍÓÚÑ ]+)",
        "etiqueta": r"\b(RELEVANTE|GACETA JUDICIAL)\b",
        "id": r"\bID:\s*([0-9]{3,})\b",
        "numero_proceso": r"\bN[ÚU]MERO DE PROCESO:\s*([0-9\-]+)\b",
        "numero_providencia": r"\bN[ÚU]MERO DE PROVIDENCIA:\s*([A-Z]{1,5}[0-9\-]+|[0-9\-]+)\b",
        "clase_actuacion": r"\bCLASE DE ACTUACI[ÓO]N:\s*(.*?)(?=\bTIPO DE PROVIDENCIA:|\bFECHA:|\bPONENTE:|\bTEMA:)",
        "tipo_providencia": r"\bTIPO DE PROVIDENCIA:\s*(SENTENCIA|AUTO|PROVIDENCIA)\b",
        "fecha": r"\bFECHA:\s*([0-9]{2}[\/\-]?[0-9]{2}[\/\-]?[0-9]{4})\b",
        "ponente": r"\bPONENTE:\s*(.*?)(?=\bTEMA:|\bEXTRACTO\b)",
        "tema_resultado": r"\bTEMA:\s*(.*?)(?=\bEXTRACTO\b|\bASUNTO\b|\bFUENTE FORMAL\b|\bFUENTE JURISPRUDENCIAL\b)",
        "extracto": r"\bEXTRACTO\s*[-–:]?\s*(.*?)(?=\bASUNTO\b|\bFUENTE FORMAL\b|\bFUENTE JURISPRUDENCIAL\b|\bPROCEDENCIA\b|\bDECISI[ÓO]N\b|$)",
    }

    data = {}
    for campo, patron in patrones.items():
        m = re.search(patron, raw, flags=re.IGNORECASE | re.DOTALL)
        data[campo] = clean_text(m.group(1)) if m else ""

    data["texto_completo"] = raw_clean
    return data

print("✓ parse_resultado corregido")

✓ parse_resultado corregido


In [9]:
# ==========================================================
# CELDA 22 — Reconstruir sets de deduplicación
# ==========================================================

df_existente = cargar_csv_seguro(CSV_RESULTADOS_PATH)

IDS_DESCARGADOS = set()
PROVIDENCIAS_DESCARGADAS = set()
PROCESOS_DESCARGADOS = set()
PDFS_DESCARGADOS = set()

if not df_existente.empty:
    if "id" in df_existente.columns:
        IDS_DESCARGADOS = set(
            df_existente["id"].dropna().astype(str).str.strip()
        )

    if "numero_providencia" in df_existente.columns:
        PROVIDENCIAS_DESCARGADAS = set(
            df_existente["numero_providencia"].dropna().astype(str).str.strip()
        )

    if "numero_proceso" in df_existente.columns:
        PROCESOS_DESCARGADOS = set(
            df_existente["numero_proceso"].dropna().astype(str).str.strip()
        )

    if "ruta_pdf" in df_existente.columns:
        PDFS_DESCARGADOS = set(
            df_existente["ruta_pdf"].dropna().astype(str).str.strip()
        )

print("✓ Índices de deduplicación reconstruidos")
print(f"IDs: {len(IDS_DESCARGADOS)}")
print(f"Providencias: {len(PROVIDENCIAS_DESCARGADAS)}")
print(f"Procesos: {len(PROCESOS_DESCARGADOS)}")
print(f"PDFs: {len(PDFS_DESCARGADOS)}")

NameError: name 'cargar_csv_seguro' is not defined

In [ ]:
# ==========================================================
# CELDA 23 — Deduplicación robusta
# ==========================================================

def ya_descargado(data: dict) -> bool:
    _id = clean_text(data.get("id", ""))
    _prov = clean_text(data.get("numero_providencia", ""))
    _proc = clean_text(data.get("numero_proceso", ""))

    if _id and _id in IDS_DESCARGADOS:
        return True
    if _prov and _prov in PROVIDENCIAS_DESCARGADAS:
        return True
    if _proc and _proc in PROCESOS_DESCARGADOS:
        return True

    return False


def marcar_descargado(data: dict, ruta_pdf: str):
    _id = clean_text(data.get("id", ""))
    _prov = clean_text(data.get("numero_providencia", ""))
    _proc = clean_text(data.get("numero_proceso", ""))
    _pdf = clean_text(ruta_pdf)

    if _id:
        IDS_DESCARGADOS.add(_id)
    if _prov:
        PROVIDENCIAS_DESCARGADAS.add(_prov)
    if _proc:
        PROCESOS_DESCARGADOS.add(_proc)
    if _pdf:
        PDFS_DESCARGADOS.add(_pdf)

print("✓ Deduplicación robusta activada")

In [ ]:
# ==========================================================
# CELDA 24 — Temas priorizados y expandidos
# ==========================================================

TEMAS_PRIORITARIOS = [
    "RESPONSABILIDAD CONTRACTUAL",
    "INTERPRETACION DE LOS CONTRATOS",
    "INCUMPLIMIENTO CONTRACTUAL",
    "RESOLUCION DEL CONTRATO",
    "COMPRAVENTA",
    "SIMULACION ABSOLUTA",
    "ERROR EN EL CONSENTIMIENTO",
    "MUTUO",
    "MANDATO",
    "LESION ENORME",
    "OBJETO DEL CONTRATO",
    "AUTONOMIA DE LA VOLUNTAD PRIVADA",
    "EXCEPCION DE CONTRATO NO CUMPLIDO",
    "VICIOS DEL CONSENTIMIENTO",
    "NULIDAD RELATIVA",
    "CAUSA DEL CONTRATO",
    "SIMULACION RELATIVA",
    "FUERZA MAYOR",
    "OBLIGACIONES A PLAZO",
    "BUENA FE OBJETIVA CONTRACTUAL",
]

VARIANTES_EXTRA = [
    "RESPONSABILIDAD CIVIL CONTRACTUAL",
    "INTERPRETACION CONTRACTUAL",
    "INTERPRETACION DE CLAUSULAS CONTRACTUALES",
    "INCUMPLIMIENTO DEL CONTRATO",
    "RESOLUCION POR INCUMPLIMIENTO",
    "PROMESA DE COMPRAVENTA",
    "CONTRATO DE COMPRAVENTA",
    "CONTRATO DE MUTUO",
    "CONTRATO DE MANDATO",
    "SIMULACION DE CONTRATO",
    "NULIDAD DEL CONTRATO",
    "CLAUSULA PENAL PECUNIARIA",
    "CONTRATO DE ARRENDAMIENTO",
    "PACTO COMISORIO",
    "MORA DEL DEUDOR",
    "OBLIGACIONES CONTRACTUALES",
    "CUMPLIMIENTO DEL CONTRATO",
    "INEJECUCION CONTRACTUAL",
    "EFECTOS DEL CONTRATO",
    "INTERPRETACION DE LA DEMANDA CONTRACTUAL",
]

TEMAS_EXPANDIDOS = list(dict.fromkeys(TEMAS_PRIORITARIOS + VARIANTES_EXTRA))

print(f"Temas expandidos: {len(TEMAS_EXPANDIDOS)}")
for i, t in enumerate(TEMAS_EXPANDIDOS, 1):
    print(f"{i:02d}. {t}")

In [ ]:
# ==========================================================
# CELDA 25 — Límite variable por tema
# ==========================================================

TEMAS_ALTO_RENDIMIENTO = {
    "RESPONSABILIDAD CONTRACTUAL",
    "INTERPRETACION DE LOS CONTRATOS",
    "INCUMPLIMIENTO CONTRACTUAL",
    "RESOLUCION DEL CONTRATO",
    "COMPRAVENTA",
    "SIMULACION ABSOLUTA",
    "ERROR EN EL CONSENTIMIENTO",
    "MUTUO",
    "MANDATO",
}

def max_por_tema_dinamico(tema: str) -> int:
    if tema in TEMAS_ALTO_RENDIMIENTO:
        return 180
    return 90

print("✓ Política dinámica de max_por_tema lista")

In [ ]:
# ==========================================================
# CELDA 26 — Ejecución masiva dinámica
# ==========================================================

async def ejecutar_lote_dinamico(
    temas: list[str],
    meta_total: int = 2000,
    headless: bool = True,
    reintentar_omitidos: bool = False,
):
    resumen = []

    pendientes = temas
    if not reintentar_omitidos:
        pendientes = [t for t in temas if t not in estado["temas_completados"]]

    log.info(f"Temas pendientes: {len(pendientes)}")

    pw, browser, ctx, page = await iniciar_contexto(headless=headless)
    try:
        for i, tema in enumerate(pendientes, 1):
            if estado["total_descargas"] >= meta_total:
                log.info("Meta global alcanzada")
                break

            max_tema = max_por_tema_dinamico(tema)

            print(f"\n===== TEMA {i}/{len(pendientes)}: {tema} =====")
            print(f"max_por_tema = {max_tema}")

            try:
                salida = await procesar_tema(page, tema, max_por_tema=max_tema)
                resumen.append(salida)
            except Exception as e:
                registrar_fallo(tema, "tema_completo_dinamico", str(e))
                resumen.append({
                    "tema": tema,
                    "descargadas": 0,
                    "procesadas": 0,
                    "errores": 1,
                })

            guardar_estado(estado)
            pd.DataFrame(resumen).to_csv(
                RUTA_TABLAS / "resumen_lote_dinamico_csj.csv",
                index=False
            )

            print("Resumen parcial:")
            print(pd.DataFrame(resumen).tail(10))
            print(f"Total descargas acumuladas: {estado['total_descargas']}")

    finally:
        await browser.close()
        await pw.stop()

    df_resumen = pd.DataFrame(resumen)
    df_resumen.to_csv(RUTA_TABLAS / "resumen_lote_dinamico_csj.csv", index=False)
    return df_resumen

print("✓ ejecutar_lote_dinamico listo")

In [ ]:
# ==========================================================
# CELDA 27 — Nueva corrida enfocada
# ==========================================================

async def main():
    df_resumen = await ejecutar_lote_dinamico(
        temas=TEMAS_EXPANDIDOS,
        meta_total=1200,
        headless=True,
        reintentar_omitidos=False,
    )

    print("\n=== RESUMEN NUEVA CORRIDA ===")
    print(df_resumen)

    print("\n=== ESTADO ACTUALIZADO ===")
    print(f"Total PDFs descargados según estado: {estado['total_descargas']}")
    print(f"Total fallos según estado: {estado['total_fallos']}")
    print(f"Metadata CSV: {CSV_RESULTADOS_PATH}")
    print(f"Fallos CSV: {CSV_FALLOS_PATH}")
    print(f"Resumen dinámico: {RUTA_TABLAS / 'resumen_lote_dinamico_csj.csv'}")

    return df_resumen

df_resumen = asyncio.get_event_loop().run_until_complete(main())

In [ ]:
# ==========================================================
# CELDA 28 — Auditoría posterior a la nueva corrida
# ==========================================================

df_meta = cargar_csv_seguro(CSV_RESULTADOS_PATH)
df_fallos = cargar_csv_seguro(CSV_FALLOS_PATH)
df_resumen_old = cargar_csv_seguro(RUTA_TABLAS / "resumen_lote_csj.csv")
df_resumen_new = cargar_csv_seguro(RUTA_TABLAS / "resumen_lote_dinamico_csj.csv")

print("=== AUDITORÍA POSTERIOR ===")
print(f"Registros metadata: {len(df_meta)}")
print(f"Fallos registrados: {len(df_fallos)}")
print(f"Temas resumen anterior: {len(df_resumen_old)}")
print(f"Temas resumen nuevo: {len(df_resumen_new)}")

if not df_meta.empty:
    print("\nTop 25 temas por volumen:")
    print(df_meta["tema_busqueda"].value_counts().head(25))

    if "numero_providencia" in df_meta.columns:
        print("\nProvidencias únicas:")
        print(df_meta["numero_providencia"].nunique())

    if "id" in df_meta.columns:
        print("\nIDs únicos:")
        print(df_meta["id"].astype(str).nunique())

    if "numero_proceso" in df_meta.columns:
        print("\nProcesos únicos:")
        print(df_meta["numero_proceso"].astype(str).nunique())